# Hypothesis 3 — T4 独立题库采集
流程只有 **配置 → 跑题 → 下载**。暂不做研究评价、不计算准确率/F1/校正分/模型排名。

默认完整运行本实验的 11,569 个请求/模型，共 12,767 个 typed decisions：
原生单选 800 母题，同题 2–5 选一 600 母题，不定项多选 200 母题。
默认模型为已验证基础环境的 Laya English 和 Multilingual；本实验专用 runner 尚待 Colab T4 实测。

使用 Colab 自带 Python/PyTorch，不访问云盘、不要求 GH_TOKEN、不从 Colab 提交 GitHub、不做 smoke。
case 或模型失败会保存日志并继续。原始回答、概率、参考答案和条件标签将供之后设计评价。
仅英文题；源标签独立人工复核仍待完成，协作者最终 H1/H2 去重清单尚未提供。


In [1]:
# 1. 配置：安装依赖、准备本实验题库（无需模型推理）
import sys, json, subprocess, hashlib
from pathlib import Path
from importlib.metadata import version
from datetime import datetime, timezone
from uuid import uuid4

MODELS = ["english", "multilingual"]
PROFILE = "full"  # 完整题库；可手动改为 pilot，只采集60道开发母题的所有配对变体
EXCLUSION_FILES = []  # 若拿到 H1/H2 最终排除清单，填入 JSON 文件路径
assert MODELS and all(m in ("english", "multilingual") for m in MODELS)
assert PROFILE in ("full", "pilot")
assert Path("/content").exists(), "请在 Colab 中执行"
assert sys.version_info >= (3, 12), "需要 Python 3.12+"

LOG_DIR = Path("/content/hypothesis3-logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)
COMMAND_NUMBER = 0

def run(args, cwd=None):
    global COMMAND_NUMBER
    args = list(map(str, args))
    COMMAND_NUMBER += 1
    log = LOG_DIR / f"command-{COMMAND_NUMBER:03d}.log"
    while log.exists():
        COMMAND_NUMBER += 1
        log = LOG_DIR / f"command-{COMMAND_NUMBER:03d}.log"
    print("+", " ".join(args), "\n日志:", log, flush=True)
    tail = []
    with log.open("w", encoding="utf-8") as stream:
        process = subprocess.Popen(
            args, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding="utf-8", errors="replace", bufsize=1)
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                stream.write(line)
                stream.flush()
                tail.append(line)
                tail = tail[-40:]
            code = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
    if code:
        raise RuntimeError(f"退出码 {code}；日志 {log}\n{''.join(tail)}")

run(["nvidia-smi"])
import torch
assert torch.cuda.is_available(), "请选择 T4 GPU 运行时"
TORCH_VERSION = version("torch")
print("复用 PyTorch:", TORCH_VERSION, "GPU:", torch.cuda.get_device_name(0))

REPO = Path("/content/system1bench")
if not REPO.exists():
    run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
         "https://github.com/Jaffe2718/system1bench.git", REPO])
    run(["git", "sparse-checkout", "set", "system1bench", "research/hypothesis3_v1"], cwd=REPO)
else:
    run(["git", "pull", "--ff-only", "origin", "main"], cwd=REPO)
    if (REPO / ".git/info/sparse-checkout").exists():
        run(["git", "sparse-checkout", "add", "research/hypothesis3_v1"], cwd=REPO)

PY = Path(sys.executable)
EXPERIMENT = REPO / "research/hypothesis3_v1"
assert (EXPERIMENT / "run.py").is_file()
CONSTRAINT = Path("/content/hypothesis3-torch-constraint.txt")
CONSTRAINT.write_text("torch==" + TORCH_VERSION)
run([PY, "-m", "pip", "install", "--disable-pip-version-check",
     "--constraint", CONSTRAINT, "-r", "requirements-colab-t4.txt"], cwd=REPO)
run([PY, "-c",
     "import sys, torch, numpy, pyarrow.parquet, laya; "
     "from importlib.metadata import version; "
     "from laya.common import build_sequence, render_options, serialize_state; "
     "assert version('torch') == sys.argv[1]; print('依赖导入成功，PyTorch 保持不变')",
     TORCH_VERSION], cwd=REPO)

BANK_REVISION = hashlib.sha256(
    (EXPERIMENT / "build.py").read_bytes() + (EXPERIMENT / "sources.json").read_bytes()).hexdigest()[:12]
BANK = Path("/content/hypothesis3-banks") / BANK_REVISION
CACHE = Path("/content/hypothesis3-cache/raw")
# 外部排除清单也进入目录标识，避免复用未经该清单过滤的旧题库。
if EXCLUSION_FILES:
    extra = hashlib.sha256(b"".join(Path(p).read_bytes() for p in EXCLUSION_FILES)).hexdigest()[:12]
    BANK = BANK.with_name(BANK.name + "-" + extra)
if BANK.exists():
    run([PY, EXPERIMENT / "build.py", "--output", BANK, "--verify"], cwd=REPO)
else:
    args = [PY, "-u", EXPERIMENT / "build.py", "--cache", CACHE, "--output", BANK]
    for filename in EXCLUSION_FILES:
        args += ["--exclude", filename]
    run(args, cwd=REPO)

# 默认题库还要与已冻结的公开构建回执逐文件比对。
if not EXCLUSION_FILES:
    receipt = json.loads((EXPERIMENT / "bank_receipt.json").read_text())
    assert hashlib.sha256((BANK / "manifest.json").read_bytes()).hexdigest() == receipt["manifest_sha256"]
    for filename, expected in receipt["files"].items():
        assert hashlib.sha256((BANK / filename).read_bytes()).hexdigest() == expected["sha256"], filename
    print("题库与冻结回执完全一致。")

SESSION = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S") + "-" + uuid4().hex[:6]
OUTPUT = Path("/content/hypothesis3-results") / SESSION
OUTPUT.mkdir(parents=True, exist_ok=True)
(OUTPUT / "configuration.json").write_text(json.dumps(
    {"models":MODELS,"profile":PROFILE,"bank":str(BANK),"evaluation_performed":False},
    ensure_ascii=False,indent=2))
print("配置完成。下一步加载模型后直接跑本实验题库。", BANK, OUTPUT)


+ nvidia-smi 
日志: /content/hypothesis3-logs/command-001.log
Mon Oct 26 13:50:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |            

In [2]:
# 2. 跑题：模型下载/加载完成后立即采集，不做 smoke，不进行研究评价
run([PY, "-u", EXPERIMENT / "run.py", "--bank", BANK, "--output", OUTPUT,
     "--models", *MODELS, "--profile", PROFILE], cwd=REPO)

status = json.loads((OUTPUT / "run_status.json").read_text())
print("运行状态（不是模型分数）:", status["status"])
for model, result in status["models"].items():
    print(model, result["status"], "已处理请求:", result.get("completed_requests", 0),
          "运行失败请求:", result.get("failed_requests", 0))
print("请继续下一格下载原始数据；评价方案之后再制定。")


+ /usr/bin/python3 -u /content/system1bench/research/hypothesis3_v1/run.py --bank /content/hypothesis3-banks/8f11eb5985d7 --output /content/hypothesis3-results/20261026-135052-0c19a6 --models english multilingual --profile full 
日志: /content/hypothesis3-logs/command-007.log

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

Fetching 5 files: 100%|██████████| 5/5 [00:09<00:00,  1.99s/it]
/content/system1bench/system1bench/laya_adapter.py:32: RuntimeWarning: laya: this checkpoint ships invalid temperatures or values outside [0.5, 5]; using choice:11+=0.10058280825614929 -> 0.5. Treat confidence from the affected entries as uncalibrated.
  self.agent = Agent(str(checkpoint), subfolder=sub or None, device="cuda")
PROGRESS english 100 / 11569 errors 0
PROGRESS english 200 / 11569 errors 0
PROGRESS english 300 / 11569 errors 0
PROGRESS english 400 / 

In [3]:
# 3. 下载：原始预测 + 题库/参考答案 + 版本/代码/日志，不包含研究评分
import shutil
from google.colab import files

(OUTPUT / "environment-pip-freeze.txt").write_text(subprocess.check_output(
    [str(PY), "-m", "pip", "freeze"], text=True))
(OUTPUT / "repository-commit.txt").write_text(subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip())
(OUTPUT / "working-tree.diff").write_text(subprocess.check_output(
    ["git", "diff", "--no-ext-diff", "HEAD"], cwd=REPO, text=True))
shutil.copytree(BANK, OUTPUT / "question_bank", dirs_exist_ok=True)
shutil.copytree(LOG_DIR, OUTPUT / "configuration-logs", dirs_exist_ok=True)
code = OUTPUT / "collection_code"
code.mkdir(exist_ok=True)
for name in ("build.py", "run.py", "sources.json", "bank_receipt.json", "PROTOCOL.md", "README.md", "NOTICE.md"):
    shutil.copyfile(EXPERIMENT / name, code / name)
archive = shutil.make_archive("/content/hypothesis3-t4-results", "zip", OUTPUT)
print("请把 ZIP 下载保存；本轮不评价，之后根据这份数据设计评价。")
files.download(archive)


请把 ZIP 下载保存；本轮不评价，之后根据这份数据设计评价。


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>